In [43]:
import pandas as pd 
import numpy as np
import sklearn

In [2]:
data = pd.read_csv("italy_airbnb_pre_cleaned.csv")

In [3]:
data.head()

,id,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,price,price_quote_price_per_night,minimum_nights,number_of_reviews,review_scores_rating,city
0,2737,41.871360,12.482150,Private room,Private room,1,1.5,NaN,59.52,59.52,31.0,5,4.80,Rome
1,11834,41.895447,12.491181,Entire rental unit,Entire home/apt,2,1.0,1.0,NaN,NaN,2.0,302,4.87,Rome
2,12398,41.925820,12.469280,Entire rental unit,Entire home/apt,3,1.0,2.0,117.13,117.13,31.0,85,4.90,Rome
3,19965,41.908230,12.452930,Entire condo,Entire home/apt,5,1.0,2.0,160.60,160.60,2.0,195,4.58,Rome
4,20534,41.889920,12.468230,Entire vacation home,Entire home/apt,4,1.0,1.0,243.67,243.67,3.0,50,4.48,Rome


In [4]:
data.shape

(70897, 14)

In [5]:
data.isnull().sum()

id                                 0
latitude                           0
longitude                          0
property_type                      0
room_type                          0
accommodates                       0
bathrooms                      11004
bedrooms                       12877
price                           5050
price_quote_price_per_night     5051
minimum_nights                    22
number_of_reviews                  0
review_scores_rating            9020
city                               0
dtype: int64

In [6]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 70897 entries, 0 to 70896
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id                           70897 non-null  int64  
 1   latitude                     70897 non-null  float64
 2   longitude                    70897 non-null  float64
 3   property_type                70897 non-null  str    
 4   room_type                    70897 non-null  str    
 5   accommodates                 70897 non-null  int64  
 6   bathrooms                    59893 non-null  float64
 7   bedrooms                     58020 non-null  float64
 8   price                        65847 non-null  float64
 9   price_quote_price_per_night  65846 non-null  float64
 10  minimum_nights               70875 non-null  float64
 11  number_of_reviews            70897 non-null  int64  
 12  review_scores_rating         61877 non-null  float64
 13  city                       

In [7]:
data = data.dropna()

In [8]:
data.isnull().sum() 

id                             0
latitude                       0
longitude                      0
property_type                  0
room_type                      0
accommodates                   0
bathrooms                      0
bedrooms                       0
price                          0
price_quote_price_per_night    0
minimum_nights                 0
number_of_reviews              0
review_scores_rating           0
city                           0
dtype: int64

In [9]:
data.info()

<class 'pandas.DataFrame'>
Index: 44946 entries, 2 to 70889
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id                           44946 non-null  int64  
 1   latitude                     44946 non-null  float64
 2   longitude                    44946 non-null  float64
 3   property_type                44946 non-null  str    
 4   room_type                    44946 non-null  str    
 5   accommodates                 44946 non-null  int64  
 6   bathrooms                    44946 non-null  float64
 7   bedrooms                     44946 non-null  float64
 8   price                        44946 non-null  float64
 9   price_quote_price_per_night  44946 non-null  float64
 10  minimum_nights               44946 non-null  float64
 11  number_of_reviews            44946 non-null  int64  
 12  review_scores_rating         44946 non-null  float64
 13  city                         449

In [10]:
data.shape

(44946, 14)

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [12]:
numeric_cols = [
    "accommodates",
    "bathrooms",
    "bedrooms",
    "minimum_nights",
    "number_of_reviews",
    "review_scores_rating"
]

In [13]:
categorical_cols = [
    "property_type",
    "room_type",
    "city"
]

In [14]:
X = data[numeric_cols + categorical_cols]
y = data["price"]

In [15]:
ct = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ), categorical_cols),
        
    ],
    remainder = "passthrough"
)

In [16]:
X_transformed = ct.fit_transform(X)

In [17]:
print(X.shape)
print(X_transformed.shape)

(44946, 9)
(44946, 78)


In [18]:
feature_names = ct.get_feature_names_out()
print(feature_names)
print(len(feature_names))

['cat__property_type_Boat' 'cat__property_type_Camper/RV'
 'cat__property_type_Casa particular' 'cat__property_type_Castle'
 'cat__property_type_Cave' 'cat__property_type_Dammuso'
 'cat__property_type_Entire bed and breakfast'
 'cat__property_type_Entire bungalow' 'cat__property_type_Entire cabin'
 'cat__property_type_Entire chalet' 'cat__property_type_Entire condo'
 'cat__property_type_Entire cottage'
 'cat__property_type_Entire guest suite'
 'cat__property_type_Entire guesthouse' 'cat__property_type_Entire home'
 'cat__property_type_Entire home/apt' 'cat__property_type_Entire loft'
 'cat__property_type_Entire place' 'cat__property_type_Entire rental unit'
 'cat__property_type_Entire serviced apartment'
 'cat__property_type_Entire townhouse'
 'cat__property_type_Entire vacation home'
 'cat__property_type_Entire villa' 'cat__property_type_Farm stay'
 'cat__property_type_Houseboat' 'cat__property_type_Private room'
 'cat__property_type_Private room in bed and breakfast'
 'cat__property_

In [19]:
from sklearn.model_selection import train_test_split


X_train , X_test , y_train , y_test = train_test_split(
    X, y, test_size=0.23, random_state=42)

In [20]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline([
    ("preprocessor", ct),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

In [21]:
rf_pipeline.fit(X_train , y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains 

In [22]:
y_pred_rf = rf_pipeline.predict(X_test)

In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [24]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

In [25]:
print("Random Forest - MAE:", mae_rf)
print("Random Forest - RMSE:", rmse_rf)
print("Random Forest - R²:", r2_rf)

Random Forest - MAE: 95.03362089883223
Random Forest - RMSE: 292.6153798742753
Random Forest - R²: -0.08261391446570188


In [26]:
from sklearn.linear_model import LinearRegression

lr_pipeline = Pipeline([
    ("preprocessor", ct),
    ("model", LinearRegression())
])

In [27]:
lr_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains 

In [28]:
y_pred_lr = lr_pipeline.predict(X_test)

In [29]:
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

In [30]:
print("Linear Regression - MAE:", mae_lr)
print("Linear Regression - RMSE:", rmse_lr)
print("Linear Regression - R²:", r2_lr)

Linear Regression - MAE: 86.9729745832952
Linear Regression - RMSE: 256.46246200948974
Linear Regression - R²: 0.16837617844724362


In [31]:
from sklearn.ensemble import GradientBoostingRegressor

gbr_pipeline = Pipeline([
    ("preprocessor", ct),
    ("model", GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

In [32]:
gbr_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains 

In [33]:
y_pred_gbr = gbr_pipeline.predict(X_test)

In [34]:
mae_gbr = mean_absolute_error(y_test, y_pred_gbr)
rmse_gbr = np.sqrt(mean_squared_error(y_test, y_pred_gbr))
r2_gbr = r2_score(y_test, y_pred_gbr)

In [35]:
print("Gradient Boosting - MAE:", mae_gbr)
print("Gradient Boosting - RMSE:", rmse_gbr)
print("Gradient Boosting - R²:", r2_gbr)

Gradient Boosting - MAE: 83.4562379107455
Gradient Boosting - RMSE: 268.37050779004124
Gradient Boosting - R²: 0.08935547129477739


In [36]:
data.head()

,id,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,price,price_quote_price_per_night,minimum_nights,number_of_reviews,review_scores_rating,city
2,12398,41.92582,12.46928,Entire rental unit,Entire home/apt,3,1.0,2.0,117.13,117.13,31.0,85,4.90,Rome
3,19965,41.90823,12.45293,Entire condo,Entire home/apt,5,1.0,2.0,160.60,160.60,2.0,195,4.58,Rome
4,20534,41.88992,12.46823,Entire vacation home,Entire home/apt,4,1.0,1.0,243.67,243.67,3.0,50,4.48,Rome
5,20587,41.88992,12.46823,Entire vacation home,Entire home/apt,4,1.0,2.0,302.50,302.50,2.0,90,4.69,Rome
6,20699,41.89294,12.50581,Private room in guesthouse,Private room,2,1.0,1.0,122.60,122.60,1.0,766,4.82,Rome


In [37]:
city_counts = data['city'].value_counts().reset_index()

city_counts.columns = ['city', 'count']

city_counts

,city,count
0,Rome,23559
1,Florence,9411
2,Venice,6080
3,Naples,5896


In [38]:
room_counts = data['room_type'].value_counts().reset_index()

room_counts.columns = ['room_type', 'count']

room_counts

,room_type,count
0,Entire home/apt,41151
1,Private room,3518
2,Hotel room,220
3,Shared room,57


In [39]:
property_counts = data['property_type'].value_counts().reset_index()

property_counts.columns = ['property_type', 'count']

property_counts


,property_type,count
0,Entire rental unit,26856
1,Entire condo,9013
2,Entire vacation home,2011
3,Entire home,1829
4,Private room in rental unit,1317
...,...,...
59,Camper/RV,1
60,Private room in resort,1
61,Houseboat,1
62,Private room in houseboat,1


In [40]:
data.to_csv("italy_airbnb_dropped.csv", index=False)

In [41]:
import pickle 

with open("model.pkl", "wb") as f:
    pickle.dump(gbr_pipeline, f)

# Save the ColumnTransformer and Encoders as well

with open("column_transformer.pkl", "wb") as f:
    pickle.dump(ct, f)


In [44]:
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("numpy:", np.__version__)
print("pickle:", pickle.format_version)

pandas: 3.0.2
scikit-learn: 1.8.0
numpy: 2.4.4
pickle: 5.0
